# Score Variation Under Random Comparison Order

TrueSkill ratings are order dependent: replaying the same comparison history in a
different order can produce different scores, even though the comparisons themselves are
identical. This notebook measures that sensitivity on the real dataset.

Each replay is a full pass over **every stored comparison** in a different random
permutation. An image's score moves between replays only because the comparisons it
takes part in landed in a different order relative to its opponents. Images with no
comparisons retain the initial rating in every replay and therefore have no order-driven
movement.

Every image in the database is measured, not a sample. Scoring all of them costs
roughly the same wall-clock as a small sample, because the runtime is dominated by the
comparison replay itself rather than by scoring the results. The image and comparison
counts for a given run are printed by the load cell below.

## How to read the axes

Every axis carries its units and a direction note, so no chart needs a legend:

| note | meaning |
|---|---|
| **higher is better** | a quality score; a bigger number is a better-rated image |
| **lower is better (more confident)** | a `sigma_uncertainty` axis; a smaller number means the scorer trusts its own estimate more |
| **lower is better (more stable, not a quality judgement)** | a movement axis (`range` or `std` across replays); a smaller number means the score held still when the order changed, which says nothing about whether the image is good |
| **more comparisons = better-estimated score** | the `comparison_count` axis; more comparisons shrink `sigma_uncertainty` |

That third row is the one worth keeping in mind. Movement is measured on a scale where
**low is good**, but it is a statement about reliability, not about image quality. A highly
rated image and a badly rated image can both be perfectly stable.

## Three score forms

The notebook shows two rating scores recalculated during every comparison-order replay,
plus the production graph position score:

| | public score | conservative TrueSkill score | position score |
|---|---|---|---|
| basis | win probability vs a default opponent | `mu_skill - 3 * sigma_uncertainty` | node position in its main graph chain |
| scale | bounded `0.0` to `1.0` | unbounded TrueSkill points | integer from `0` to `100` |
| replayed? | yes | yes | no; production graph value |

The public score compresses movement near 0 and 1. The conservative score includes an
uncertainty penalty. Position score is a separate graph-derived production metric; it is
not recalculated for each random comparison order and is excluded from replay movement
statistics. Its values and distribution are shown separately.

## A note on the word "std"

There is no "public sigma" in this module. All three score values are single numbers, so none has a spread of its own. Confidence
exists only as
`Rating.sigma_uncertainty`, one value per rating.

Every `std` and `range` in this notebook is measured **across the replays** for a single
image, so it describes how much that image's score moves when the comparison order
changes. That is order sensitivity, not confidence, and the two are different things.
Section 4 prints the correlation between them for the current data rather than assuming
a value, so the distinction can be checked instead of trusted.

Nothing here writes to the database. All scores are recalculated in memory, including
the production baseline, which is replayed in stored order rather than read from
`cache.db`.

All tqdm progress bars use `delay=3` matching the codebase style.

In [ ]:
import random
import time
from typing import TypedDict

import matplotlib.image as mpimg
from matplotlib.axes import Axes
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tqdm import tqdm as tqdm_desc

from comfyui_image_scorer.application.services.graph_service import CrystalGraph
from comfyui_image_scorer.domain.analysis.trueskill import (
    Rating,
    public_score_from_rating,
    replay_ratings,
    trueskill_score_from_rating,
)
from comfyui_image_scorer.infrastructure.persistence.comparisons_repository import (
    list_links,
)
from comfyui_image_scorer.infrastructure.persistence.images_repository import list_nodes
from comfyui_image_scorer.domain.ports.repository import ComparisonRow, ImageRow
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path

from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    configure_package_logging,
    get_logger,
)

# 20 = INFO. __name__ is __main__ in a notebook and configure_package_logging
# sets that logger's level, so anything higher would drop every info record.
configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)


class BandSummary(TypedDict):
    row: int
    label: str
    population: int
    default_deviation: float
    random_deviation: float
    gap: float
    deviations: np.ndarray
    spread: float
    level: float


def rating_for_image(ratings: dict[str, tuple[Rating, int]], filename: str) -> Rating:
    """Keep images without comparisons at the unmodified prior rating."""
    if filename not in ratings:
        return Rating()
    return ratings[filename][0]


## 1. Load Data

In [ ]:
image_rows: list[ImageRow] = list_nodes()
comparison_rows: list[ComparisonRow] = list_links()

graph = CrystalGraph()
graph.rebuild_from_database(images=image_rows, comparisons=comparison_rows)
position_scores: np.ndarray = np.array(
    [graph.get_node(row["filename"]).position_score for row in image_rows],
    dtype=np.int16,
)

filenames: list[str] = [row["filename"] for row in image_rows]
comparison_counts: np.ndarray = np.array(
    [int(row["comparison_count"]) for row in image_rows], dtype=np.int32
)

logger.info(
    "production position_score min/median/max: %d / %d / %d",
    int(position_scores.min()),
    int(np.median(position_scores)),
    int(position_scores.max()),
)
logger.info(
    "Loaded %d images and %d comparisons",
    len(filenames),
    len(comparison_rows),
)
# Images with no comparisons remain in the analysis at their initial rating.
logger.info(
    "comparison_count min/median/max: %d / %d / %d",
    int(comparison_counts.min()),
    int(np.median(comparison_counts)),
    int(comparison_counts.max()),
)
logger.info(
    "comparison_count percentiles p50/p90/p99/p99.9: %d / %d / %d / %d",
    int(np.percentile(comparison_counts, 50)),
    int(np.percentile(comparison_counts, 90)),
    int(np.percentile(comparison_counts, 99)),
    int(np.percentile(comparison_counts, 99.9)),
)
# Where the bulk of the data would sit on a full linear axis running to the maximum.
# Whether that is cramped or perfectly fine depends entirely on the shape of the
# distribution, so it is measured here instead of asserted. Section 6 reads these back
# when it picks its own cutoff.
maximum_comparison_count: int = int(comparison_counts.max())
axis_share_at_p90: float = (
    float(np.percentile(comparison_counts, 90)) / maximum_comparison_count
)
logger.info(
    "a full linear axis to the maximum (%d) would place 90%% of images within the "
    "left %.2f%% of its width",
    maximum_comparison_count,
    100.0 * axis_share_at_p90,
)


## 2. Replay Every Image Through Every Order

Builds replay matrices for the public and conservative TrueSkill scores, plus a
production baseline replayed in stored comparison-id order. The graph-derived position
score is calculated once from the production graph and is not part of these replays.

In [ ]:
# Number of random permutations of the comparison history replayed per run. A sample
# standard deviation is only pinned to about 1/sqrt(2*(n-1)) relative error, so raise
# this to tighten the per-image spread estimates at the cost of linear time.
REPLAY_COUNT: int = 50

# Seeds the permutation shuffles and deterministic chart jitter, so a whole run is
# reproducible from this one value.
SEED: int = 12300

public_scores: np.ndarray = np.empty((REPLAY_COUNT, len(filenames)), dtype=np.float32)
trueskill_scores: np.ndarray = np.empty(
    (REPLAY_COUNT, len(filenames)), dtype=np.float32
)
# Rating.sigma_uncertainty is kept as its own matrix. The trueskill score is
# mu - 3 * sigma, so folding sigma into one score hides the confidence term
# entirely; section 7 measures it directly.
sigma_uncertainties: np.ndarray = np.empty(
    (REPLAY_COUNT, len(filenames)), dtype=np.float32
)

replay_start = time.perf_counter()
for replay_index in tqdm_desc(
    range(REPLAY_COUNT), desc="random replays", unit="replay", delay=3
):
    random.seed(SEED + replay_index)
    ratings: dict[str, tuple[Rating, int]] = replay_ratings(
        comparison_rows, order="random"
    )
    replay_ratings_by_filename = {
        name: rating_for_image(ratings, name) for name in filenames
    }
    public_scores[replay_index] = [
        public_score_from_rating(replay_ratings_by_filename[name]) for name in filenames
    ]
    trueskill_scores[replay_index] = [
        trueskill_score_from_rating(replay_ratings_by_filename[name])
        for name in filenames
    ]
    sigma_uncertainties[replay_index] = [
        replay_ratings_by_filename[name].sigma_uncertainty for name in filenames
    ]

# The production baseline is replayed through the same path as the random replays
# rather than read from cache.db, so the two are directly comparable.
default_ratings = replay_ratings(comparison_rows, order="default")


default_public_score = np.array(
    [
        public_score_from_rating(rating_for_image(default_ratings, name))
        for name in filenames
    ],
    dtype=np.float32,
)
default_trueskill_score = np.array(
    [
        trueskill_score_from_rating(rating_for_image(default_ratings, name))
        for name in filenames
    ],
    dtype=np.float32,
)
default_sigma_uncertainty = np.array(
    [rating_for_image(default_ratings, name).sigma_uncertainty for name in filenames],
    dtype=np.float32,
)

logger.info(
    "Replayed %d random orders over %d images (%.1fs)",
    REPLAY_COUNT,
    len(filenames),
    time.perf_counter() - replay_start,
)


## 3. Per-Image Statistics

One row per image: how far its score travels across the replays, and where the
production score sits inside that spread.

In [ ]:
per_image_mean_public = public_scores.mean(axis=0)
per_image_mean_trueskill = trueskill_scores.mean(axis=0)
per_image_public_std = public_scores.std(axis=0, ddof=1)
per_image_min_public = public_scores.min(axis=0)
per_image_max_public = public_scores.max(axis=0)
per_image_range_public = per_image_max_public - per_image_min_public
per_image_trueskill_std = trueskill_scores.std(axis=0, ddof=1)
per_image_range_trueskill = trueskill_scores.max(axis=0) - trueskill_scores.min(axis=0)

# Midranks place the production score within each image's random spread; ties split
# their weight evenly between the ranks below and above the production score.
less_than_default = np.sum(public_scores < default_public_score, axis=0)
equal_to_default = np.sum(public_scores == default_public_score, axis=0)
default_rank_public = (less_than_default + (0.5 * equal_to_default)) / REPLAY_COUNT

# sigma_uncertainty is the module's own confidence. Only the conservative score folds
# it into mu_skill - 3 * sigma_uncertainty; it is tracked separately here.
per_image_uncertainty_mean = sigma_uncertainties.mean(axis=0)
per_image_uncertainty_std = sigma_uncertainties.std(axis=0, ddof=1)
per_image_uncertainty_range = sigma_uncertainties.max(axis=0) - sigma_uncertainties.min(
    axis=0
)

# Highly-ranked images can settle at exactly zero spread across replays, so the
# denominator is masked rather than clipped; those images become NaN and drop out of
# the finite-only stats.
safe_public_std = np.where(per_image_public_std > 0.0, per_image_public_std, np.nan)
default_z_public = (default_public_score - per_image_mean_public) / safe_public_std

logger.info(
    "public score span %.3f to %.3f | conservative score span %.2f to %.2f "
    "| sigma_uncertainty span %.3f to %.3f",
    float(public_scores.min()),
    float(public_scores.max()),
    float(trueskill_scores.min()),
    float(trueskill_scores.max()),
    float(sigma_uncertainties.min()),
    float(sigma_uncertainties.max()),
)


## 4. Headline Numbers

In [ ]:
summary_table = pd.DataFrame(
    [
        {
            "metric": "public score std",
            "mean": float(per_image_public_std.mean()),
            "median": float(np.median(per_image_public_std)),
            "p90": float(np.percentile(per_image_public_std, 90)),
            "max": float(per_image_public_std.max()),
        },
        {
            "metric": "public range",
            "mean": float(per_image_range_public.mean()),
            "median": float(np.median(per_image_range_public)),
            "p90": float(np.percentile(per_image_range_public, 90)),
            "max": float(per_image_range_public.max()),
        },
        {
            "metric": "trueskill score std",
            "mean": float(per_image_trueskill_std.mean()),
            "median": float(np.median(per_image_trueskill_std)),
            "p90": float(np.percentile(per_image_trueskill_std, 90)),
            "max": float(per_image_trueskill_std.max()),
        },
        {
            "metric": "production position score (not replayed)",
            "mean": float(position_scores.mean()),
            "median": float(np.median(position_scores)),
            "p90": float(np.percentile(position_scores, 90)),
            "max": float(position_scores.max()),
        },
        {
            "metric": "trueskill range",
            "mean": float(per_image_range_trueskill.mean()),
            "median": float(np.median(per_image_range_trueskill)),
            "p90": float(np.percentile(per_image_range_trueskill, 90)),
            "max": float(per_image_range_trueskill.max()),
        },
    ]
)
display(summary_table.round(4))

# Reference only: without tied scores, exchangeable orders give REPLAY_COUNT + 1
# equally likely ranks. Ties change that support, so this is not an inferential test.
possible_ranks = np.arange(REPLAY_COUNT + 1) / REPLAY_COUNT
outside_5_95_reference: float = float(
    np.mean((possible_ranks < 0.05) | (possible_ranks > 0.95))
)
outside_1_99_reference: float = float(
    np.mean((possible_ranks < 0.01) | (possible_ranks > 0.99))
)

finite_z = default_z_public[np.isfinite(default_z_public)]
logger.info(
    "production score outside its own random band: [5,95] %.1f%%, [1,99] %.1f%% "
    "(uniform untied ranks would give %.1f%% / %.1f%%)",
    100.0 * float(np.mean((default_rank_public < 0.05) | (default_rank_public > 0.95))),
    100.0 * float(np.mean((default_rank_public < 0.01) | (default_rank_public > 0.99))),
    100.0 * outside_5_95_reference,
    100.0 * outside_1_99_reference,
)
logger.info(
    "z spread of the production score: std %.3f, |z|>2 %.1f%%, |z|>3 %.1f%% "
    "(a normal distribution would give 1.000 / 4.6%% / 0.3%%)",
    float(finite_z.std()),
    100.0 * float(np.mean(np.abs(finite_z) > 2.0)),
    100.0 * float(np.mean(np.abs(finite_z) > 3.0)),
)
logger.info(
    "movement vs confidence: corr(mean sigma_uncertainty, public std) = %.3f, "
    "conservative TrueSkill std = %.3f; confidence and order sensitivity are "
    "related but distinct",
    float(np.corrcoef(per_image_uncertainty_mean, per_image_public_std)[0, 1]),
    float(np.corrcoef(per_image_uncertainty_mean, per_image_trueskill_std)[0, 1]),
)


Each image contributes the midrank of its production score inside its own distribution
of random-order scores. A rank near `0.0` means the production score is below nearly every
replay; near `1.0`, above nearly every one. Equal scores receive a midrank, so images whose
scores never move land at `0.5` rather than being pushed to one end.

The dashed line is the expected bar height **if the production order were exchangeable
with the random permutations**. This is a descriptive reference, not a significance test:
images share comparisons and are therefore not independent observations. Read the overall
shape as a summary of how the stored order compares with randomized orders, not as a
p-value or pass/fail result.

In [ ]:
# One bin per replay gives a readable rank histogram without implying independent samples.
BIN_COUNT: int = REPLAY_COUNT
rank_histogram, _ = np.histogram(default_rank_public, bins=BIN_COUNT, range=(0.0, 1.0))

# Under exchangeability, the production order would be another random draw, so ranks
# would be approximately uniform. This is a visual reference only: image ranks are
# dependent because images share comparisons.
expected_per_bin: float = len(filenames) / BIN_COUNT

figure, axis = plt.subplots(figsize=(12, 5.5))
axis.axhline(
    expected_per_bin,
    color="tab:red",
    linestyle="--",
    linewidth=2,
    label=f"uniform reference under exchangeability ({expected_per_bin:.0f} images/bin)",
)
axis.bar(
    range(BIN_COUNT),
    rank_histogram,
    color="tab:blue",
    edgecolor="black",
    width=0.82,
    label="observed images",
)
axis.set_xticks(np.arange(0, BIN_COUNT + 1, max(BIN_COUNT // 10, 1)))
axis.set_xticklabels([f"{index / BIN_COUNT:.1f}" for index in axis.get_xticks()])
axis.set_xlabel(
    "midrank of public production score among random replays (fraction, 0-1)\n"
    "higher = production score sits higher against its own replays"
)
axis.set_ylabel("images (count)")
axis.set_title(
    "Production-order score ranks within each image's random spread", fontsize=13
)
axis.grid(axis="y", linestyle=":", alpha=0.4)
axis.legend(loc="upper left", fontsize=9)
figure.tight_layout()
plt.show()

logger.info(
    "rank histogram uses %d bins; the uniform reference is descriptive only because "
    "images share comparisons and are not independent",
    BIN_COUNT,
)

# Compare production and random-order means for replay-derived scores. Position score
# is shown once because it belongs to the production graph and is not replayed.
score_distribution_panels = (
    (
        "public score",
        default_public_score,
        per_image_mean_public,
        "public score (0-1, higher is better)",
        "tab:blue",
    ),
    (
        "conservative TrueSkill score",
        default_trueskill_score,
        per_image_mean_trueskill,
        "mu_skill - 3 * sigma_uncertainty (TrueSkill points)",
        "tab:orange",
    ),
)

figure, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for axis, (score_name, production, random_mean, unit_label, color) in zip(
    axes[:2], score_distribution_panels
):
    options = {"range": (0.0, 1.0)} if score_name == "public score" else {}
    axis.hist(
        production,
        bins=60,
        alpha=0.65,
        color=color,
        label="production order",
        **options,
    )
    axis.hist(
        random_mean,
        bins=60,
        alpha=0.65,
        color="tab:gray",
        label=f"mean of {REPLAY_COUNT} random orders",
        **options,
    )
    axis.set_title(score_name)
    axis.set_xlabel(unit_label)
    axis.set_ylabel("images (count)")
    axis.grid(axis="y", linestyle=":", alpha=0.4)
    axis.legend(fontsize=8)

axes[2].hist(position_scores, bins=np.arange(-0.5, 101.5, 2), color="tab:green")
axes[2].axvline(position_scores.mean(), color="black", linestyle="--", linewidth=2)
axes[2].set_title("production position score")
axes[2].set_xlabel("position score (0-100, higher is better)")
axes[2].set_ylabel("images (count)")
axes[2].grid(axis="y", linestyle=":", alpha=0.4)
figure.suptitle(
    "Rating-score distributions by order and production position-score distribution",
    fontsize=13,
)
figure.tight_layout()
plt.show()


def tail_share(values: np.ndarray) -> float:
    return float(np.mean((values < 0.05) | (values > 0.95)))


logger.info(
    "public score spread: production sd %.4f against %.4f for the mean of %d random "
    "orders (ratio %.3f); images beyond 0.05 or 0.95 are %.2f%% against %.2f%%",
    float(default_public_score.std()),
    float(per_image_mean_public.std()),
    REPLAY_COUNT,
    float(default_public_score.std() / per_image_mean_public.std()),
    100.0 * tail_share(default_public_score),
    100.0 * tail_share(per_image_mean_public),
)


## 6. Does Movement Depend On How Many Comparisons An Image Has?

**What is on each axis**

- **x: how many comparisons the image has taken part in.** A count straight from the
  database, so 5 means it was compared five times.
- **y: how much the score moved.** A high dot means that image's score is unstable and
  depends on the order the comparisons arrived in. A low dot means it barely cares. This
  is a measure of movement across the random orders, not a confidence measure.

Two ways of measuring that movement are shown side by side, because they answer slightly
different questions:

- **range** (`max - min`) is the full span the score covered. It answers *how far could
  this score have ended up anywhere?*
- **std** is the typical wobble around its own average. It answers *how much does it
  usually move?* and is far less sensitive to one unlucky replay.

So the chart answers one question: **do images with more comparisons move less?**

**How to read it**

- Every image is its own dot. Nothing is grouped or averaged except the black median
  line, which exists only because the full cloud needs a reference. Bands holding fewer
  than `MIN_BAND_IMAGES` images are skipped, so the line stops where the data stops
  supporting it rather than being drawn through noise.
- `comparison_count` is a whole number, so each dot is nudged sideways by up to half a
  step to stop thousands of dots stacking into a single vertical stripe.
- The x axis is **linear** and stops at `MAX_TREND_COMPARISON_COUNT`. Whether that cutoff
  is needed at all depends on the shape of the data: the load cell measures where the
  bulk of the images would land on a full linear axis, and if that share comes out large
  for your data then this can be raised to the full maximum without crowding anything.
  The chart also logs how many images fall beyond the cutoff and how much they move, so
  the cutoff never hides anything silently.
- Each score form has its own column and y axis because the public score, raw `mu_skill`,
  and conservative TrueSkill score use different scales. **Do not compare panel heights
  across columns or rows**; only the shape within a panel is meaningful.


In [ ]:
# The comparison_count axis stops here so the scale can stay linear and readable.
MAX_TREND_COMPARISON_COUNT: int = 25
MIN_BAND_IMAGES: int = 50
band_edges = np.unique(np.linspace(0, MAX_TREND_COMPARISON_COUNT + 1, 21).astype(int))
in_axis_range = comparison_counts <= MAX_TREND_COMPARISON_COUNT
jitter = np.random.default_rng(SEED).uniform(-0.5, 0.5, size=len(filenames))
jittered_counts = comparison_counts + jitter

# Position score is not included: it is not recomputed during random replays.
measures = (
    (
        per_image_range_public,
        "public score range (max - min)",
        "0-1 public score units",
        "tab:blue",
    ),
    (
        per_image_public_std,
        "public score std (typical wobble)",
        "0-1 public score units",
        "tab:blue",
    ),
    (
        per_image_range_trueskill,
        "conservative TrueSkill range (max - min)",
        "TrueSkill points (mu - 3 sigma)",
        "tab:orange",
    ),
    (
        per_image_trueskill_std,
        "conservative TrueSkill std (typical wobble)",
        "TrueSkill points (mu - 3 sigma)",
        "tab:orange",
    ),
)

figure, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
for axis, (values, measure_label, unit_label, color) in zip(axes.flat, measures):
    axis.scatter(
        jittered_counts[in_axis_range],
        values[in_axis_range],
        s=4,
        alpha=0.12,
        color=color,
        edgecolors="none",
    )
    band_centers: list[float] = []
    band_medians: list[float] = []
    for low, high in zip(band_edges[:-1], band_edges[1:]):
        in_band = (
            in_axis_range & (comparison_counts >= low) & (comparison_counts < high)
        )
        if np.count_nonzero(in_band) >= MIN_BAND_IMAGES:
            band_centers.append(float(np.median(comparison_counts[in_band])))
            band_medians.append(float(np.median(values[in_band])))
    axis.plot(band_centers, band_medians, color="black", linewidth=2.5, label="median")
    axis.set_xlim(0, MAX_TREND_COMPARISON_COUNT + 1)
    axis.set_xlabel(
        "comparisons on the image (count; more comparisons = better-estimated score)"
    )
    axis.set_ylabel(unit_label)
    axis.set_title(
        f"{measure_label}\nlower is better (more stable, not a quality judgement)",
        fontsize=11,
    )
    axis.grid(linestyle=":", alpha=0.4)
    axis.legend(loc="upper right")

figure.suptitle(
    f"Replay-derived score movement over {REPLAY_COUNT} random comparison orders",
    fontsize=13,
)
figure.tight_layout()
plt.show()

beyond_axis = comparison_counts > MAX_TREND_COMPARISON_COUNT
logger.info(
    "%d images (%0.1f%%) sit beyond the axis because they have more than %d "
    "comparisons; median public range %.4f / std %.4f, conservative range %.3f / std %.3f",
    int(beyond_axis.sum()),
    100.0 * float(beyond_axis.mean()),
    MAX_TREND_COMPARISON_COUNT,
    float(np.median(per_image_range_public[beyond_axis])),
    float(np.median(per_image_public_std[beyond_axis])),
    float(np.median(per_image_range_trueskill[beyond_axis])),
    float(np.median(per_image_trueskill_std[beyond_axis])),
)


## 7. sigma_uncertainty, Measured On Its Own

This section is about `Rating.sigma_uncertainty`, the module's confidence estimate. It is
not a score: the public score does not include sigma, while the conservative TrueSkill
score folds it into `mu_skill - 3 * sigma_uncertainty`. Position score is graph-derived
and is not part of the TrueSkill uncertainty calculation.

Two ratings with the same `mu_skill` but different confidence get different conservative
TrueSkill scores, so part of that score's movement is the uncertainty term.

**What is on each axis**

- **x: how many comparisons the image has taken part in** (count).
- **Left panel, y: average sigma** across the random orders, in TrueSkill points. A low
  value means the scorer is confident about that image. This is the confidence curve: more
  comparisons should mean lower sigma.
- **Right panel, y: how much sigma itself moved** across those orders, as a standard
  deviation in TrueSkill points. This answers a different question: not *is the scorer
  confident* but *does the scorer's confidence depend on the order the comparisons arrived
  in*. An image whose sigma is stable across orders sits at zero regardless of how
  confident the scorer is overall.

Both panels plot one dot per image with a black median line, on the same terms as section 6.

**The second figure asks whether reordering changes confidence.** Each row is one
comparison-count band, and every sigma value is centred on its own band's average taken
across *both* orders, so zero is that band's average and the two markers are directly
comparable:

- the pale points are **individual orderings** of sampled images in that band, with a
  small vertical nudge so the density stays visible;
- the **red diamond** is the production order;
- the **black dash** is the mean of the random orders.

The distance between the two markers is the *systematic* effect, which the bar chart on the
right quantifies per band. The width of the point cloud is the *per-ordering* variation.

Keep those two apart. Individual sigma values can move, but movement that averages out is
not the same as confidence getting better or worse.

In [ ]:
# Same cutoff and band rules as section 6, so the two sections can be read side by side.
MIN_BAND_IMAGES: int = 50

band_edges = np.unique(np.linspace(0, MAX_TREND_COMPARISON_COUNT + 1, 21).astype(int))
in_axis_range = comparison_counts <= MAX_TREND_COMPARISON_COUNT

# Reuse the seed so the sideways nudge matches section 6 dot for dot.
jitter = np.random.default_rng(SEED).uniform(-0.5, 0.5, size=len(filenames))
jittered_counts = comparison_counts + jitter


def median_line(axis: Axes, values: np.ndarray) -> None:
    band_centers: list[float] = []
    band_medians: list[float] = []
    for low, high in zip(band_edges[:-1], band_edges[1:]):
        in_band = (
            in_axis_range & (comparison_counts >= low) & (comparison_counts < high)
        )
        if np.count_nonzero(in_band) >= MIN_BAND_IMAGES:
            band_centers.append(float(np.median(comparison_counts[in_band])))
            band_medians.append(float(np.median(values[in_band])))
    axis.plot(band_centers, band_medians, color="black", linewidth=2.5, label="median")
    axis.set_xlim(0, MAX_TREND_COMPARISON_COUNT + 1)
    axis.set_xlabel(
        "comparisons on the image (count; more comparisons = better-estimated score)"
    )
    axis.grid(linestyle=":", alpha=0.4)
    axis.legend(loc="upper right")


uncertainty_panels = (
    (
        per_image_uncertainty_mean,
        "average sigma_uncertainty",
        "TrueSkill points",
        "lower is better (more confident)",
    ),
    (
        per_image_uncertainty_std,
        "sigma_uncertainty std across replays",
        "TrueSkill points",
        "lower is better (more stable, not a quality judgement)",
    ),
)

figure, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True)
for axis, (values, measure_label, unit_label, direction) in zip(
    axes, uncertainty_panels
):
    axis.scatter(
        jittered_counts[in_axis_range],
        values[in_axis_range],
        s=4,
        alpha=0.12,
        color="tab:purple",
        edgecolors="none",
    )
    median_line(axis, values)
    axis.set_ylabel(unit_label)
    axis.set_title(f"{measure_label}\n{direction}", fontsize=12)

figure.suptitle(
    "Rating.sigma_uncertainty on its own, one dot per image\n"
    f"measured over {REPLAY_COUNT} random comparison orders",
    fontsize=13,
)
figure.tight_layout()
plt.show()

# The trueskill score is mu - 3 * sigma, so sigma moving contributes directly to that
# score's movement. This is the size of that contribution rather than an assumed share.
sigma_contribution = 3.0 * per_image_uncertainty_range
logger.info(
    "sigma's contribution to trueskill-score movement: median 3 * sigma range is %.3f "
    "against a median trueskill-score range of %.3f, so roughly %.0f%% of that score's "
    "movement is the confidence term rather than skill",
    float(np.median(sigma_contribution)),
    float(np.median(per_image_range_trueskill)),
    100.0
    * float(
        np.median(sigma_contribution / np.maximum(per_image_range_trueskill, 1e-9))
    ),
)

beyond_axis = comparison_counts > MAX_TREND_COMPARISON_COUNT
logger.info(
    "%d images (%0.1f%%) sit beyond the axis; their average sigma is %.3f and their "
    "sigma std is %.4f",
    int(beyond_axis.sum()),
    100.0 * float(beyond_axis.mean()),
    float(np.median(per_image_uncertainty_mean[beyond_axis])),
    float(np.median(per_image_uncertainty_std[beyond_axis])),
)

# ── Does random ordering make the scorer more or less confident? ─────────────────
#
# sigma is a conservative quantity, so reordering could leave it alone on average while
# individual images still move. Each row below is one comparison-count band.
#
# Every sigma value is centred on its own band's mean taken across both orders, so zero
# is that band's average and the production marker and the random mean sit on one scale.
# Points are individual orderings; the distance between the two markers is the systematic
# part, while the width of the cloud is the per-ordering variation.
COMPARISON_BAND_EDGES: np.ndarray = np.unique(
    np.linspace(0, MAX_TREND_COMPARISON_COUNT + 1, 11).astype(int)
)
BAND_IMAGE_SAMPLE: int = 200


def band_label(low: int, high: int) -> str:
    """Render a half-open [low, high) band of integer comparison counts."""
    return str(low) if high - 1 == low else f"{low}-{high - 1}"


band_rows: list[tuple[str, np.ndarray]] = []
for low, high in zip(COMPARISON_BAND_EDGES[:-1], COMPARISON_BAND_EDGES[1:]):
    in_band = in_axis_range & (comparison_counts >= low) & (comparison_counts < high)
    if np.count_nonzero(in_band) >= MIN_BAND_IMAGES:
        band_rows.append((band_label(int(low), int(high)), in_band))

# Deterministic image sample per band. Every ordering of every sampled image is kept,
# because the spread of those points is the question.
sampler = np.random.default_rng(SEED)
band_summaries: list[BandSummary] = []
for row, (label, in_band) in enumerate(band_rows):
    band_default = default_sigma_uncertainty[in_band]
    band_random = sigma_uncertainties[:, in_band]
    band_centre = float(np.concatenate([band_default, band_random.ravel()]).mean())

    band_images = np.flatnonzero(in_band)
    if band_images.size > BAND_IMAGE_SAMPLE:
        band_images = np.sort(
            sampler.choice(band_images, size=BAND_IMAGE_SAMPLE, replace=False)
        )
    sampled_random = sigma_uncertainties[:, band_images]
    band_summaries.append(
        {
            "row": row,
            "label": label,
            "population": int(np.count_nonzero(in_band)),
            "default_deviation": float(band_default.mean()) - band_centre,
            "random_deviation": float(band_random.mean()) - band_centre,
            "gap": float(band_random.mean()) - float(band_default.mean()),
            "deviations": (sampled_random - band_centre).ravel(),
            "spread": float(sampled_random.std(axis=0, ddof=1).mean()),
            "level": float(band_random.mean()),
        }
    )

# A small vertical nudge keeps the points from collapsing into flat streaks, so the
# density along x stays visible. It moves nothing in the x direction.
spread_limit = float(
    np.percentile(
        np.abs(np.concatenate([summary["deviations"] for summary in band_summaries])),
        99,
    )
)
spread_limit = max(spread_limit, 1e-6)

figure, (strip_axis, gap_axis) = plt.subplots(
    1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [3, 2]}
)

for summary in band_summaries:
    row_index = int(summary["row"])
    row = float(row_index)
    deviations = np.asarray(summary["deviations"])
    vertical = row + np.random.default_rng(SEED + row_index).uniform(
        -0.28, 0.28, size=deviations.size
    )
    strip_axis.scatter(
        deviations,
        vertical,
        s=2,
        alpha=0.03,
        color="tab:purple",
        edgecolors="none",
    )
    strip_axis.plot(
        [float(summary["default_deviation"])],
        [row],
        marker="D",
        markersize=7,
        color="tab:red",
        linestyle="none",
        label="production order" if row_index == 0 else None,
    )
    strip_axis.plot(
        [float(summary["random_deviation"])],
        [row],
        marker="_",
        markersize=15,
        markeredgewidth=2.5,
        color="black",
        linestyle="none",
        label="mean of random orders" if row_index == 0 else None,
    )

strip_axis.axvline(0.0, color="grey", linewidth=1.5, linestyle=":")
strip_axis.set_yticks([float(summary["row"]) for summary in band_summaries])
strip_axis.set_yticklabels(
    [f"{summary['label']}  (n={summary['population']:,})" for summary in band_summaries]
)
# Lowest comparison count at the top, matching how the other charts read left to right.
strip_axis.set_ylim(len(band_summaries) - 0.5, -0.5)
# Clipped to the bulk so a handful of extreme orderings cannot flatten the markers onto
# the zero line, which is the comparison this chart exists to show.
strip_axis.set_xlim(-spread_limit, spread_limit)
strip_axis.set_xlabel(
    "sigma_uncertainty deviation from this band's average (TrueSkill points)\n"
    "clipped to the 99th percentile; lower is better (more confident)"
)
strip_axis.set_ylabel("comparisons on the image (band)")
strip_axis.set_title(
    "Every individual ordering, centred on its own band's average", fontsize=12
)
strip_axis.grid(axis="x", linestyle=":", alpha=0.4)
strip_axis.legend(loc="lower right", fontsize=9)

gaps = [float(summary["gap"]) for summary in band_summaries]
positions = np.arange(len(band_summaries))
gap_axis.axhline(0.0, color="grey", linewidth=1.5)
gap_axis.bar(
    positions,
    gaps,
    # orange where random ordering leaves the scorer more confident, blue where less
    color=["tab:orange" if gap < 0 else "tab:blue" for gap in gaps],
    edgecolor="black",
    width=0.7,
)
gap_axis.set_xticks(positions)
gap_axis.set_xticklabels([str(summary["label"]) for summary in band_summaries])
gap_axis.set_xlabel("comparisons on the image (band)")
gap_axis.set_ylabel(
    "random mean minus production mean\n(TrueSkill points, below zero = more confident)"
)
gap_axis.set_title("Systematic effect of random ordering, by band", fontsize=12)
gap_axis.grid(axis="y", linestyle=":", alpha=0.4)

figure.suptitle(
    "Does reordering the comparisons make the scorer more or less confident?",
    fontsize=13,
)
figure.tight_layout()
plt.show()

overall_default = float(default_sigma_uncertainty.mean())
overall_random = float(sigma_uncertainties.mean())
relative_shift = 100.0 * (overall_random - overall_default) / overall_default
bands_down = sum(1 for gap in gaps if gap < 0)
relative_spread = float(
    np.median(
        [
            float(summary["spread"]) / float(summary["level"])
            for summary in band_summaries
        ]
    )
)
logger.info(
    "averaged over every image and every order: production sigma %.4f, random sigma "
    "%.4f, so random ordering shifts the average by %+.4f points (%+.1f%%)",
    overall_default,
    overall_random,
    overall_random - overall_default,
    relative_shift,
)
logger.info(
    "the shift has no consistent direction: %d of %d bands move down and %d move up. "
    "Ordering does change individual sigma values, by about %.1f%% of the sigma level "
    "per image, but it does not move the population average in one direction",
    bands_down,
    len(gaps),
    len(gaps) - bands_down,
    100.0 * relative_spread,
)


## 8. How The Replay-Derived Scores Differ

The same images are compared using the public and conservative TrueSkill scores. Both are
recomputed for each random order, but they use different scales and transformations: the
public score saturates near 0 and 1, while the conservative score is linear in
`mu_skill - 3 * sigma_uncertainty`. Compare the curves to see how those transformations
affect measured order sensitivity. Position score is not included because it belongs to
the production graph and is not recalculated during random replays.

In [ ]:
# The median line is a reading aid; score bands thinner than this hold too few
# images for their median to mean anything.
MIN_SCORE_BAND_IMAGES: int = 50

distance_from_middle = np.abs(per_image_mean_public - 0.5)
figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True)
variance_panels = (
    (
        axes[0],
        per_image_public_std,
        "public score std",
        "0-1 public score units",
        "tab:blue",
    ),
    (
        axes[1],
        per_image_trueskill_std,
        "conservative TrueSkill score std",
        "TrueSkill points (mu - 3 sigma)",
        "tab:orange",
    ),
)
score_band_edges = np.linspace(0.0, 1.0, 21)
for axis, stds, measure_label, unit_label, color in variance_panels:
    axis.plot(
        per_image_mean_public,
        stds,
        linestyle="none",
        marker=".",
        markersize=2,
        alpha=0.03,
        color=color,
    )
    band_centers: list[float] = []
    band_medians: list[float] = []
    for low, high in zip(score_band_edges[:-1], score_band_edges[1:]):
        in_band = (per_image_mean_public >= low) & (per_image_mean_public < high)
        if np.count_nonzero(in_band) >= MIN_SCORE_BAND_IMAGES:
            band_centers.append(float(np.median(per_image_mean_public[in_band])))
            band_medians.append(float(np.median(stds[in_band])))
    axis.plot(band_centers, band_medians, color="black", linewidth=2.5)
    axis.set_xlabel("mean public score across replays (0-1, higher is better)")
    axis.set_ylabel(f"{unit_label}\nlower is better (more stable)")
    axis.set_title(
        f"{measure_label}\ncorr(distance from 0.5, std) = {np.corrcoef(distance_from_middle, stds)[0, 1]:.2f}",
        fontsize=12,
    )
    axis.grid(linestyle=":", alpha=0.4)
figure.suptitle(
    "Replay-derived score movement against position on the public-score scale",
    fontsize=13,
)
figure.tight_layout()
plt.show()


## 9. Distributions

The movement histograms cover only scores recalculated during random replays. Position
score is shown as a production-graph distribution in section 5, not as replay movement.

In [ ]:
histogram_panels = (
    (per_image_public_std, "public score std", "0-1 public score units", "tab:blue"),
    (
        per_image_range_public,
        "public range (max - min)",
        "0-1 public score units",
        "tab:blue",
    ),
    (
        per_image_trueskill_std,
        "conservative TrueSkill score std",
        "TrueSkill points (mu - 3 sigma)",
        "tab:orange",
    ),
    (
        per_image_range_trueskill,
        "conservative TrueSkill range (max - min)",
        "TrueSkill points (mu - 3 sigma)",
        "tab:orange",
    ),
)
figure, axes = plt.subplots(2, 2, figsize=(15, 8))
for axis, (values, measure_label, unit_label, color) in zip(
    axes.flat, histogram_panels
):
    axis.hist(values, bins=80, color=color, edgecolor="none", alpha=0.85)
    axis.axvline(
        values.mean(), color="black", linestyle="--", linewidth=2, label="mean"
    )
    axis.set_xlabel(
        f"{measure_label}\n[{unit_label}]\nlower is better (more stable, not a quality judgement)"
    )
    axis.set_ylabel("images (count)")
    axis.grid(axis="y", linestyle=":", alpha=0.4)
    axis.legend(loc="upper right")
figure.suptitle(
    f"Replay movement for {len(filenames):,} images over {REPLAY_COUNT} random orders",
    fontsize=13,
)
figure.tight_layout()
plt.show()


## 10. Example Images Across The Full Score Range

Images are ordered by mean public score and sampled at even positions, so the examples
sit as far apart as the distribution allows. Each thumbnail shows its production
position score alongside replay-derived rating movement.

In [ ]:
# Thumbnails rendered across the public-score range.
EXAMPLE_COUNT: int = 10

by_mean_public = np.argsort(per_image_mean_public)
example_positions = np.linspace(0, len(by_mean_public) - 1, EXAMPLE_COUNT).astype(int)
example_indices = by_mean_public[example_positions]

thumbnail_columns: int = 5
thumbnail_rows: int = int(np.ceil(EXAMPLE_COUNT / thumbnail_columns))

figure, axes = plt.subplots(
    thumbnail_rows,
    thumbnail_columns,
    figsize=(4 * thumbnail_columns, 3.2 * thumbnail_rows),
    squeeze=False,
)
for slot, image_index in enumerate(example_indices):
    axis = axes.flat[slot]
    name = filenames[image_index]
    path = find_image_path(name)
    if path is None:
        axis.text(
            0.5,
            0.5,
            f"not found: {name}",
            ha="center",
            va="center",
            fontsize=8,
        )
    else:
        axis.imshow(mpimg.imread(path))
    axis.set_title(
        f"cc={comparison_counts[image_index]}  "
        f"position score {position_scores[image_index]}\n"
        f"public {per_image_min_public[image_index]:.3f}-{per_image_max_public[image_index]:.3f}  "
        f"std {per_image_public_std[image_index]:.4f} / "
        f"{per_image_trueskill_std[image_index]:.3f}",
        fontsize=9,
    )
    axis.set_xticks([])
    axis.set_yticks([])

for slot in range(EXAMPLE_COUNT, thumbnail_rows * thumbnail_columns):
    axes.flat[slot].axis("off")

figure.suptitle(
    f"{EXAMPLE_COUNT} images spread across the public-score range "
    "(position score / public std / conservative std)",
    fontsize=13,
)
figure.tight_layout()
plt.show()


## 11. Most Movement, Least Movement

Image galleries for the largest and smallest **public-score movement ranges**. Each image
shows its production graph position score, its public-score range across random replays,
and its comparison count. The least-movement gallery includes images with at least
`SETTLED_MIN_COMPARISONS` comparisons.

In [ ]:
# Rows in each gallery and comparison floor for the least-movement gallery.
TABLE_ROWS: int = 10
SETTLED_MIN_COMPARISONS: int = 10

most_movement_indices = np.argsort(per_image_range_public)[-TABLE_ROWS:][::-1]
settled_candidates = np.flatnonzero(comparison_counts >= SETTLED_MIN_COMPARISONS)
least_movement_indices = settled_candidates[
    np.argsort(per_image_range_public[settled_candidates])[:TABLE_ROWS]
]


def show_movement_gallery(title: str, image_indices: np.ndarray) -> None:
    figure, axes = plt.subplots(2, 5, figsize=(18, 8), squeeze=False)
    for axis, image_index in zip(axes.flat, image_indices):
        index = int(image_index)
        filename = filenames[index]
        path = find_image_path(filename)
        if path is None:
            axis.text(0.5, 0.5, f"not found: {filename}", ha="center", va="center")
        else:
            axis.imshow(mpimg.imread(path))
        axis.set_title(
            f"position score: {position_scores[index]}\n"
            f"public movement range: {per_image_range_public[index]:.4f} | "
            f"comparisons: {comparison_counts[index]}",
            fontsize=9,
        )
        axis.set_xticks([])
        axis.set_yticks([])

    for axis in axes.flat[len(image_indices) :]:
        axis.axis("off")

    figure.suptitle(title, fontsize=13)
    figure.tight_layout()
    plt.show()


show_movement_gallery(
    f"Most public-score movement (top {TABLE_ROWS} by replay range)",
    most_movement_indices,
)
show_movement_gallery(
    f"Least public-score movement (at least {SETTLED_MIN_COMPARISONS} comparisons)",
    least_movement_indices,
)
